# P02 — Network Data Import

Load CEPII GeoDist and Gravity datasets, convert to Arrow, and validate country code alignment with QoG.

**Sources:**
- [CEPII GeoDist](https://www.cepii.fr/CEPII/en/bdd_modele/bdd_modele_item.asp?id=6) — bilateral distances, contiguity, language, colonial ties
- [CEPII Gravity](https://www.cepii.fr/CEPII/en/bdd_modele/bdd_modele_item.asp?id=8) — annual bilateral trade flows + controls

In [ ]:
include("phase2/functions/load_phase2.jl")

## 1. GeoDist — Static Bilateral Network

In [ ]:
geodist = load_geodist()

In [ ]:
describe(geodist)

In [ ]:
# Sample: edges for a single country
filter(r -> r.iso_o == "USA", geodist) |> df -> sort(df, :distw) |> df -> first(df, 10)

## 2. GeoDist — Country Metadata

In [ ]:
geo_countries = load_geo_countries()

In [ ]:
describe(geo_countries)

## 3. Gravity — Annual Bilateral Trade Panel

In [ ]:
gravity = load_gravity()

In [ ]:
describe(gravity)

In [ ]:
# Trade data coverage by year (BACI trade flow)
year_coverage = combine(
    groupby(gravity, :year),
    :tradeflow_baci => (x -> sum(.!ismissing.(x))) => :n_baci,
    :tradeflow_comtrade_o => (x -> sum(.!ismissing.(x))) => :n_comtrade,
    :tradeflow_imf_o => (x -> sum(.!ismissing.(x))) => :n_imf,
    nrow => :n_pairs
)
sort!(year_coverage, :year)
first(year_coverage, 10)

In [ ]:
last(year_coverage, 10)

## 4. Country Code Alignment with QoG

In [ ]:
alignment = validate_country_alignment(geodist, gravity)

In [ ]:
alignment.summary

## 5. Verify Arrow Files

In [ ]:
# Verify round-trip: load from Arrow and compare
net = load_network_data()
println("GeoDist:           $(nrow(net.geodist)) rows × $(ncol(net.geodist)) cols")
println("Geo Countries:     $(nrow(net.geo_countries)) rows × $(ncol(net.geo_countries)) cols")
println("Gravity:           $(nrow(net.gravity)) rows × $(ncol(net.gravity)) cols")
println("Gravity Countries: $(nrow(net.gravity_countries)) rows × $(ncol(net.gravity_countries)) cols")

In [ ]:
# Arrow file sizes vs raw
for (label, path) in [
    ("cepii_geodist.arrow", PATH_GEODIST_ARROW),
    ("cepii_geo_countries.arrow", PATH_GEO_COUNTRIES_ARROW),
    ("cepii_gravity.arrow", PATH_GRAVITY_ARROW),
    ("cepii_gravity_countries.arrow", PATH_GRAVITY_COUNTRIES_ARROW),
]
    sz = filesize(path) / 1024^2
    println("    $(rpad(label, 35)) $(round(sz, digits=1)) MB")
end